# Phase 2: exploring the images

**CRISP-DM stage:** Data understanding

**Runs on:** Kaggle, with the GPU and internet switched off, because no neural network runs and nothing is installed. The private dataset `revathyh/mvtec-ad2-sheet-metal-vial` is attached.

**Figures with dataset images:** files named `gallery_*` hold real dataset images, so they stay on my laptop and out of git. Each figure is saved and closed, not displayed, so no image ends up in the notebook's log.

## 1. Find the data

Same search as the Phase 1 notebook, see its part 1 for why we search and do not type a fixed path.

In [ ]:
from pathlib import Path

INPUT_FOLDER = Path("/kaggle/input")
PRODUCT_NAMES = ["sheet_metal", "vial"]

product_folders = {}

for train_folder in INPUT_FOLDER.rglob("train"):
    product_folder = train_folder.parent
    if product_folder.name in PRODUCT_NAMES:
        product_folders[product_folder.name] = product_folder

for product_name in PRODUCT_NAMES:
    if product_name not in product_folders:
        raise FileNotFoundError("Could not find the folder for " + product_name)
    print(product_name, "folder:", product_folders[product_name])

## 2. Helper functions

**Matplotlib** is Python's standard plotting library. A figure holds a grid of panels called axes, and each image goes in one panel.

- `pick_evenly` chooses files spread through a sorted folder. This shows more variety than the first few files, and it repeats exactly with no randomness.
- `draw_image` draws one greyscale image. `vmin=0` and `vmax=255` fix the brightness scale, so images can be compared. Matplotlib's default stretches each image's own range, which would make a dark image look normal.
- `contour` draws a line where the mask's values cross a level. 127.5 is halfway between 0 (no defect) and 255 (defect), so the line follows the defect's edge. An outline keeps the defect visible, where a filled overlay would hide it.

In [ ]:
import matplotlib.pyplot as plt
import numpy
from PIL import Image

# Sharp enough for the report, at a reasonable file size
FIGURE_DPI = 150


def pick_evenly(paths, how_many):
    """Return up to how_many paths, spread evenly through the list."""
    if len(paths) <= how_many:
        return paths
    picked = []
    step = len(paths) / how_many
    for position in range(how_many):
        picked.append(paths[int(position * step)])
    return picked


def read_image(path):
    """Read a PNG file into a table of numbers."""
    with Image.open(path) as image:
        return numpy.array(image)


def draw_image(axis, image_path, title, mask_path=None):
    """Draw one greyscale image in one panel, with its mask outlined if given."""
    axis.imshow(read_image(image_path), cmap="gray", vmin=0, vmax=255)
    if mask_path is not None:
        axis.contour(read_image(mask_path), levels=[127.5], colors="red", linewidths=1)
    axis.set_title(title, fontsize=10)
    axis.axis("off")


def mask_path_for(product_folder, image_path):
    """Return the mask file that belongs to a defective test image."""
    mask_name = image_path.stem + "_mask.png"
    return product_folder / "test_public" / "ground_truth" / "bad" / mask_name

## 3. Sample gallery

Three rows per product: good training images, good validation images, and defective test images with the mask outlined in red. There are 4 images per row, enough to see variety while staying legible. The defective row uses `regular` lighting, so it shows the defect and not a lighting change.

The figure size differs per product, because Sheet Metal is four times wider than tall and Vial is taller than wide.

**Limit:** the figure is shrunk to fit, so the tiniest defects may be hard to see, even with an outline.

In [ ]:
IMAGES_PER_ROW = 4
GALLERY_SIZES = {"sheet_metal": (24, 6), "vial": (16, 17)}

for product_name in PRODUCT_NAMES:
    product_folder = product_folders[product_name]

    train_paths = sorted((product_folder / "train" / "good").glob("*.png"))
    validation_paths = sorted((product_folder / "validation" / "good").glob("*.png"))
    defective_paths = sorted((product_folder / "test_public" / "bad").glob("*_regular.png"))

    rows = [
        ("train", pick_evenly(train_paths, IMAGES_PER_ROW)),
        ("validation", pick_evenly(validation_paths, IMAGES_PER_ROW)),
        ("defective test", pick_evenly(defective_paths, IMAGES_PER_ROW)),
    ]

    # squeeze=False keeps the panels in a table of rows and columns
    figure, axes = plt.subplots(3, IMAGES_PER_ROW, figsize=GALLERY_SIZES[product_name], squeeze=False)

    for row_number in range(3):
        row_name, paths = rows[row_number]
        for column_number in range(IMAGES_PER_ROW):
            axis = axes[row_number][column_number]
            # A row can have fewer images than panels on a small test sample
            if column_number >= len(paths):
                axis.axis("off")
                continue
            image_path = paths[column_number]
            mask_path = None
            if row_name == "defective test":
                mask_path = mask_path_for(product_folder, image_path)
            draw_image(axis, image_path, row_name + ": " + image_path.name, mask_path)

    figure.suptitle("MVTec AD 2, " + product_name + " (CC BY-NC-SA 4.0). Red outline: defect mask")
    file_name = "gallery_" + product_name + ".png"
    figure.savefig(file_name, dpi=FIGURE_DPI, bbox_inches="tight")
    # Closing without showing keeps dataset images out of the notebook's log
    plt.close(figure)
    print("Saved", file_name)

## 4. Scene check

File names suggest each number in `test_public` is one scene photographed under several lighting conditions. Here we look at defective scene `000` under every condition, with its masks outlined. This is one scene per product, so it can support the idea but cannot prove it for every scene.

In [ ]:
SCENE_NUMBER = "000"
SCENE_COLUMNS = {"sheet_metal": 2, "vial": 4}
SCENE_SIZES = {"sheet_metal": (24, 10), "vial": (16, 11)}

for product_name in PRODUCT_NAMES:
    product_folder = product_folders[product_name]
    scene_paths = sorted((product_folder / "test_public" / "bad").glob(SCENE_NUMBER + "_*.png"))

    columns = SCENE_COLUMNS[product_name]
    # Enough rows to hold every image, rounding up
    number_of_rows = (len(scene_paths) + columns - 1) // columns
    figure, axes = plt.subplots(number_of_rows, columns, figsize=SCENE_SIZES[product_name], squeeze=False)

    for row_number in range(number_of_rows):
        for column_number in range(columns):
            axis = axes[row_number][column_number]
            position = row_number * columns + column_number
            if position >= len(scene_paths):
                axis.axis("off")
                continue
            image_path = scene_paths[position]
            draw_image(axis, image_path, image_path.name, mask_path_for(product_folder, image_path))

    figure.suptitle("MVTec AD 2, " + product_name + ", scene " + SCENE_NUMBER + " (CC BY-NC-SA 4.0). Red outline: defect mask")
    file_name = "gallery_scene_check_" + product_name + ".png"
    figure.savefig(file_name, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(figure)
    print("Saved", file_name, "with", len(scene_paths), "images")

## 5. Public test set summary

This reads file names only, so no image is opened. It reports good and defective counts, scene numbers, lighting conditions, and whether the data labels defect types.

**Flag-everything baseline:** imagine a useless system that calls every image defective. It catches every defect but rejects every good part. With a mostly defective test set, it still scores well, so a method's F1 only means something if it beats this number. It uses only the counts, with no model.

In [ ]:
summary_lines = []

for product_name in PRODUCT_NAMES:
    test_folder = product_folders[product_name] / "test_public"
    good_paths = sorted((test_folder / "good").glob("*.png"))
    defective_paths = sorted((test_folder / "bad").glob("*.png"))

    good_count = len(good_paths)
    defective_count = len(defective_paths)
    total_count = good_count + defective_count

    summary_lines.append("")
    summary_lines.append("=== " + product_name)
    summary_lines.append("good images: " + str(good_count))
    summary_lines.append("defective images: " + str(defective_count))
    summary_lines.append("defective share: " + str(round(100 * defective_count / total_count, 1)) + " percent")

    # A file name such as 000_shift_1.png holds a scene number and a lighting condition
    good_scenes = []
    defective_scenes = []
    condition_counts = {}

    for path in good_paths:
        scene, condition = path.stem.split("_", 1)
        if scene not in good_scenes:
            good_scenes.append(scene)
        if condition not in condition_counts:
            condition_counts[condition] = [0, 0]
        condition_counts[condition][0] = condition_counts[condition][0] + 1

    for path in defective_paths:
        scene, condition = path.stem.split("_", 1)
        if scene not in defective_scenes:
            defective_scenes.append(scene)
        if condition not in condition_counts:
            condition_counts[condition] = [0, 0]
        condition_counts[condition][1] = condition_counts[condition][1] + 1

    summary_lines.append("good scene numbers: " + str(len(good_scenes)))
    summary_lines.append("defective scene numbers: " + str(len(defective_scenes)))

    summary_lines.append("lighting condition: good images, defective images")
    for condition in sorted(condition_counts):
        counts = condition_counts[condition]
        summary_lines.append("  " + condition + ": " + str(counts[0]) + ", " + str(counts[1]))

    # A defect type label would appear as a subfolder or in the file names
    subfolders = []
    for path in (test_folder / "bad").iterdir():
        if path.is_dir():
            subfolders.append(path.name)
    summary_lines.append("subfolders inside test_public/bad: " + str(len(subfolders)) + " " + str(subfolders))
    summary_lines.append("distinct name endings: " + ", ".join(sorted(condition_counts)))

    # Calling everything defective means no defect is missed, so recall is 1
    precision = defective_count / total_count
    recall = 1
    f1 = 2 * precision * recall / (precision + recall)
    accuracy = defective_count / total_count
    summary_lines.append("flag-everything baseline:")
    summary_lines.append("  accuracy: " + str(round(accuracy, 3)))
    summary_lines.append("  precision: " + str(round(precision, 3)))
    summary_lines.append("  recall: " + str(recall))
    summary_lines.append("  F1: " + str(round(f1, 3)))

# Numbers only, with no image content, so this file can be committed
with open("test_set_summary.txt", "w") as summary_file:
    for line in summary_lines:
        summary_file.write(line + "\n")

for line in summary_lines:
    print(line)

## 6. Split the public test set into half A and half B

**CRISP-DM stage:** Data preparation. This was planned for Phase 3. It moved here so that defect measurements can use half A only, and half B can never influence a choice.

Half A is for practice and for every choice that needs defect examples. Half B is the final exam, used only to report final results.

- **By scene number:** all lighting versions of one part go into the same half. Otherwise the exam would hold parts already seen.
- **Good and defective separately:** each half gets a fair share of both. Defect types are not labelled in the files, so they cannot be balanced.
- **Odd numbers:** an extra defective part goes to half B and an extra good part goes to half A, so the two halves come out as even as possible. For Vial this gives 70 photos in each half. The catch is that half B then has only 2 good Vial parts.
- **Seed 42:** a seed is a starting number that makes a shuffle repeat exactly. The value itself does not matter. The split is run once and never retried for a nicer result.

A library tool such as scikit-learn's splitters could do this too. A short shuffle is easier to explain line by line and does the same job here.

In [ ]:
import csv
import random

# Any fixed number repeats the same shuffle. It is never changed to get a nicer split
SPLIT_SEED = 42
random.seed(SPLIT_SEED)


def scene_of(file_name):
    """Return the scene number from a name such as 000_shift_1.png."""
    return file_name.split("_", 1)[0]


def split_scenes(scenes, label):
    """Shuffle scene numbers and share them out as evenly as possible."""
    shuffled = list(scenes)
    random.shuffle(shuffled)
    half_a_size = len(shuffled) // 2
    # An extra good part goes to half A and an extra defective part to half B,
    # because this makes the two halves as even in size as possible
    if label == "good":
        half_a_size = len(shuffled) - half_a_size
    return shuffled[:half_a_size], shuffled[half_a_size:]


def write_file_list(file_name, rows):
    """Save one half as a table of file names, with no image content."""
    with open(file_name, "w", newline="") as list_file:
        writer = csv.writer(list_file)
        writer.writerow(["product", "label", "scene", "file_name"])
        for row in rows:
            writer.writerow(row)


half_a_rows = []
half_b_rows = []

for product_name in PRODUCT_NAMES:
    test_folder = product_folders[product_name] / "test_public"
    for label in ["good", "bad"]:
        paths = sorted((test_folder / label).glob("*.png"))

        scenes = []
        for path in paths:
            scene = scene_of(path.name)
            if scene not in scenes:
                scenes.append(scene)

        half_a_scenes, half_b_scenes = split_scenes(scenes, label)

        half_a_images = 0
        half_b_images = 0
        for path in paths:
            scene = scene_of(path.name)
            row = [product_name, label, scene, path.name]
            if scene in half_a_scenes:
                half_a_rows.append(row)
                half_a_images = half_a_images + 1
            else:
                half_b_rows.append(row)
                half_b_images = half_b_images + 1

        print(product_name, label, "half A:", len(half_a_scenes), "parts,", half_a_images, "images",
              "| half B:", len(half_b_scenes), "parts,", half_b_images, "images")

# A safety check: the same part must never appear in both halves
half_b_parts = []
for row in half_b_rows:
    half_b_parts.append((row[0], row[1], row[2]))
for row in half_a_rows:
    if (row[0], row[1], row[2]) in half_b_parts:
        raise ValueError("A part is in both halves: " + str(row))

write_file_list("half_a_files.csv", half_a_rows)
write_file_list("half_b_files.csv", half_b_rows)
print("Saved half_a_files.csv and half_b_files.csv")

## 7. Measure every defect in half A

**CRISP-DM stage:** Data understanding. Only half A is read here, so half B stays sealed.

A mask is a black-and-white copy of a photo where white marks the defect. One mask can show several separate defects, so we find each separate white shape.

- `ndimage.label` gives every separate white shape its own number.
- `structure` says which pixels count as touching. The 3 by 3 block of ones includes corners. The tool's default counts only side-by-side touching, which would break a diagonal scratch into many tiny defects. anomalib's code for its AU-PRO score uses OpenCV's `connectedComponents`, whose default also counts corners, so this matches.
- `ndimage.find_objects` gives the smallest box around each shape, which gives its width and height.

The small test first uses a made-up mask with a known answer, so a mistake in the counting shows up straight away.

**Repeated photos:** one defect appears once per lighting condition, so the number of defects is larger than the number of physical parts. Every photo is kept because some masks differ between lighting conditions.

**Limit:** width and height come from a box drawn around the defect. A thin diagonal scratch has a large box, so it can look wider than it really is.

In [ ]:
from scipy import ndimage

# A 3 by 3 block of ones means pixels touching at corners count as one defect
TOUCHING_RULE = numpy.ones((3, 3))


def find_defects(mask):
    """Return the area, width and height of each separate white shape in a mask."""
    numbered_mask, number_of_defects = ndimage.label(mask > 0, structure=TOUCHING_RULE)
    boxes = ndimage.find_objects(numbered_mask)

    defects = []
    for position in range(number_of_defects):
        box = boxes[position]
        row_range = box[0]
        column_range = box[1]
        # Shapes are numbered from 1. A box may also hold part of a neighbouring shape
        area = numpy.sum(numbered_mask[box] == position + 1)
        defects.append({
            "area": int(area),
            "width": column_range.stop - column_range.start,
            "height": row_range.stop - row_range.start,
        })
    return defects


# Made-up mask with a known answer: a 2 by 2 square, a 3-pixel diagonal and a single pixel
test_mask = numpy.zeros((6, 6))
test_mask[0:2, 0:2] = 255
test_mask[3, 3] = 255
test_mask[4, 4] = 255
test_mask[5, 5] = 255
test_mask[0, 5] = 255

test_areas = []
for defect in find_defects(test_mask):
    test_areas.append(defect["area"])
if sorted(test_areas) != [1, 3, 4]:
    raise ValueError("Defect counting is wrong: " + str(test_areas))
print("Counting test passed, areas found:", sorted(test_areas))

defect_rows = []
image_sizes = {}

for row in half_a_rows:
    product_name, label, scene, file_name = row
    # Good images have no masks, so only defective images are measured
    if label != "bad":
        continue

    image_path = product_folders[product_name] / "test_public" / "bad" / file_name
    mask = read_image(mask_path_for(product_folders[product_name], image_path))
    height, width = mask.shape
    image_sizes[product_name] = (width, height)
    image_area = width * height

    lighting = Path(file_name).stem.split("_", 1)[1]
    for defect in find_defects(mask):
        defect_rows.append({
            "product": product_name,
            "file_name": file_name,
            "scene": scene,
            "lighting": lighting,
            "area_pixels": defect["area"],
            "area_percent": 100 * defect["area"] / image_area,
            "width": defect["width"],
            "height": defect["height"],
        })

with open("defect_sizes.csv", "w", newline="") as sizes_file:
    writer = csv.DictWriter(sizes_file, fieldnames=list(defect_rows[0].keys()))
    writer.writeheader()
    for defect_row in defect_rows:
        writer.writerow(defect_row)

print("Saved defect_sizes.csv with", len(defect_rows), "defects")

## 8. How small do defects become when an image is shrunk?

This is arithmetic on the measured sizes. No image is actually shrunk, and no image size is chosen here.

If an image becomes 16 times narrower and 16 times shorter, a defect's area becomes 16 × 16 = 256 times smaller.

- **256 and 512 wide** are the two sizes in the question.
- **Keep shape:** width and height shrink by the same amount, so the part is not distorted.
- **Squash to square:** the image becomes 256 by 256 or 512 by 512. Width and height shrink by different amounts. This is included because the shape choice is not made until Phase 3.

A defect below 1 pixel of area, or narrower than 1 pixel in either direction, may disappear when shrunk.

In [ ]:
TARGET_SIZES = [256, 512]

size_lines = []

for product_name in PRODUCT_NAMES:
    areas = []
    percents = []
    parts = []
    images = []
    product_rows = []
    for defect_row in defect_rows:
        if defect_row["product"] == product_name:
            product_rows.append(defect_row)
            areas.append(defect_row["area_pixels"])
            percents.append(defect_row["area_percent"])
            if defect_row["scene"] not in parts:
                parts.append(defect_row["scene"])
            if defect_row["file_name"] not in images:
                images.append(defect_row["file_name"])

    width, height = image_sizes[product_name]

    size_lines.append("")
    size_lines.append("=== " + product_name + ", half A only")
    size_lines.append("original image: " + str(width) + " x " + str(height))
    size_lines.append("defective parts: " + str(len(parts)))
    size_lines.append("defective images: " + str(len(images)))
    size_lines.append("separate defects found: " + str(len(areas)))
    size_lines.append("area in pixels, smallest: " + str(min(areas)))
    size_lines.append("area in pixels, middle (median): " + str(numpy.median(areas)))
    size_lines.append("area in pixels, largest: " + str(max(areas)))
    size_lines.append("area in percent of image, smallest: " + str(round(min(percents), 5)))
    size_lines.append("area in percent of image, middle (median): " + str(round(float(numpy.median(percents)), 5)))
    size_lines.append("area in percent of image, largest: " + str(round(max(percents), 5)))

    for target in TARGET_SIZES:
        # The two ways of shrinking: same scale both ways, or squashed into a square
        shrink_options = [
            ("keep shape, " + str(target) + " wide", target / width, target / width),
            ("squash to " + str(target) + " x " + str(target), target / width, target / height),
        ]
        for option_name, width_scale, height_scale in shrink_options:
            below_one_area = 0
            below_one_side = 0
            for defect_row in product_rows:
                new_area = defect_row["area_pixels"] * width_scale * height_scale
                new_width = defect_row["width"] * width_scale
                new_height = defect_row["height"] * height_scale
                if new_area < 1:
                    below_one_area = below_one_area + 1
                if new_width < 1 or new_height < 1:
                    below_one_side = below_one_side + 1

            size_lines.append(option_name + ":")
            size_lines.append("  one shrunk pixel covers " + str(round(1 / (width_scale * height_scale), 1)) + " original pixels")
            size_lines.append("  smallest defect becomes " + str(round(min(areas) * width_scale * height_scale, 3)) + " pixels")
            size_lines.append("  middle defect becomes " + str(round(float(numpy.median(areas)) * width_scale * height_scale, 3)) + " pixels")
            size_lines.append("  defects below 1 pixel of area: " + str(below_one_area) + " of " + str(len(areas)))
            size_lines.append("  defects narrower than 1 pixel in width or height: " + str(below_one_side) + " of " + str(len(areas)))

# Numbers only, with no image content, so this file can be committed
with open("defect_size_summary.txt", "w") as summary_file:
    for line in size_lines:
        summary_file.write(line + "\n")

for line in size_lines:
    print(line)

## 9. Defect-size histogram

A histogram is a bar chart of how many defects fall into each size range.

- **Stretched scale:** each step along the bottom is 10 times bigger (1, 10, 100, 1,000...). Defect sizes vary hugely, and an ordinary scale would squash all the tiny ones into one bar.
- **Four bars per step of 10:** enough to show the shape without single defects looking like a pattern.
- **Same scale in both panels,** so the two products can be compared.
- **Dashed lines** mark how many original pixels fit into one shrunk pixel at 256 and 512 wide, keeping the shape. A defect left of a line would be smaller than one pixel at that size.

The figure holds numbers only, with no dataset images, so it can be committed.

In [ ]:
from matplotlib.ticker import MaxNLocator

# From 1 pixel up to 10 million, which is larger than either whole image
SIZE_BINS = numpy.logspace(0, 7, 29)

BAR_COLOUR = "#2a78d6"
LINE_COLOUR = "#3a3a38"
# Different heights stop the two line labels from overlapping
LABEL_HEIGHTS = {256: 0.97, 512: 0.87}

figure, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

for position in range(len(PRODUCT_NAMES)):
    product_name = PRODUCT_NAMES[position]
    axis = axes[position]

    areas = []
    parts = []
    for defect_row in defect_rows:
        if defect_row["product"] == product_name:
            areas.append(defect_row["area_pixels"])
            if defect_row["scene"] not in parts:
                parts.append(defect_row["scene"])

    axis.hist(areas, bins=SIZE_BINS, color=BAR_COLOUR, edgecolor="white", linewidth=1.5)
    axis.set_xscale("log")
    axis.set_ylabel("Number of defects")
    axis.set_title(product_name + ": " + str(len(areas)) + " defects from " + str(len(parts)) + " parts, half A only", loc="left")
    # You cannot have part of a defect, so the side axis uses whole numbers
    axis.yaxis.set_major_locator(MaxNLocator(integer=True))
    axis.grid(axis="y", color="#e4e4e0", linewidth=0.8)
    axis.set_axisbelow(True)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

    width, height = image_sizes[product_name]
    for target in TARGET_SIZES:
        # How many original pixels fit into one shrunk pixel, keeping the shape
        one_pixel_area = (width / target) * (width / target)
        axis.axvline(one_pixel_area, color=LINE_COLOUR, linestyle="--", linewidth=1)
        # Places the label at the top of the panel, whatever the bar heights are
        axis.text(one_pixel_area, LABEL_HEIGHTS[target], " 1 pixel at " + str(target) + " wide",
                  transform=axis.get_xaxis_transform(), fontsize=9, color=LINE_COLOUR, va="top")

axes[1].set_xlabel("Defect area in original pixels (each step is 10 times bigger)")
figure.tight_layout()
figure.savefig("defect_size_histogram.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(figure)
print("Saved defect_size_histogram.png")